# convT-kernel-axis-swap — ex2: construct a ConvT2d weight from a Conv2d weight via axis swap

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `convT-kernel-axis-swap`. Running the final beacon cell reports progress against the `CNN: ConvT kernel axis swap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT kernel axis swap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-kernel-axis-swap`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-kernel-axis-swap"
DD_SUBTOPIC = "CNN: ConvT kernel axis swap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvT kernel axis swap — quick refresher

`nn.Conv2d.weight` has shape `(OC, IC, KH, KW)`.
`nn.ConvTranspose2d.weight` has shape `(IC, OC, KH, KW)`.

The first two axes are SWAPPED relative to one another — this is the root of every "my weights don't fit" bug when porting kernels between conv and convT.

**This drill (ex2) vs ex1.** ex1 *introspected* the two layouts (read each weight.shape, label what axis 0 means). ex2 *constructs* — given a Conv2d weight tensor, build the equivalent ConvT2d weight tensor by performing the axis-0/axis-1 swap, and verify the constructed tensor loads cleanly into a `nn.ConvTranspose2d` with the right `in_channels` / `out_channels`.

### Exercise 2 — construct a ConvT2d weight from a Conv2d weight via axis swap

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Conv2d→ConvT2d weight axis swap (`transpose(0, 1)` on the `(OC, IC, KH, KW)` weight) to construct a valid ConvT2d weight tensor and load it into a `nn.ConvTranspose2d` whose in_channels/out_channels match the original Conv2d's IC/OC.
> Keywords: axis-swap, weight-construction, convT, transpose
> ```

**KCs targeted:** `convT-weight-axis-order`, `conv-vs-convT-layout`

Implement `ex2_conv_to_convT_weight(conv_weight)`.

Given a Conv2d weight tensor `conv_weight` of shape `(OC, IC, KH, KW)`, return the equivalent ConvT2d weight tensor of shape `(IC, OC, KH, KW)` (axes 0 and 1 swapped, spatial axes untouched).

**Rules.**
1. Use `.transpose(0, 1).contiguous()` — `contiguous()` matters because `nn.ConvTranspose2d` will fail to accept a non-contiguous weight via `.weight.data.copy_(...)` if shapes don't match.
2. Spatial axes (axis 2, 3) MUST be unchanged — this is not a kernel-flip operation; only the channel axes swap.
3. The returned tensor must load into a `nn.ConvTranspose2d(IC, OC, kernel_size=(KH, KW))` whose `weight.shape` matches.

Inputs:
- `conv_weight`: `(OC, IC, KH, KW)` float tensor (e.g. from a `nn.Conv2d.weight`).

Output: `(IC, OC, KH, KW)` contiguous float tensor.

In [ ]:
def ex2_conv_to_convT_weight(conv_weight: Tensor) -> Tensor:
    """Swap (OC, IC) -> (IC, OC) to convert Conv2d weight to ConvT2d weight."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn
    OC, IC, KH, KW = 4, 2, 3, 5
    rng = t.Generator().manual_seed(0)
    conv_w = t.randn(OC, IC, KH, KW, generator=rng)
    ct_w = ex2_conv_to_convT_weight(conv_w)
    assert ct_w.shape == (IC, OC, KH, KW), f'shape {tuple(ct_w.shape)} != {(IC,OC,KH,KW)}'
    assert ct_w.dtype == conv_w.dtype, f'dtype changed: {ct_w.dtype}'
    assert ct_w.is_contiguous(), 'output must be contiguous'

    # Cell-wise verification — swapped channel pair, identical spatial slice.
    for oc in range(OC):
        for ic in range(IC):
            assert t.equal(ct_w[ic, oc], conv_w[oc, ic]), (
                f'spatial slice (ic={ic}, oc={oc}) mismatch'
            )

    # Round-trip — swap back must equal the original.
    round_trip = ex2_conv_to_convT_weight(ct_w)
    assert round_trip.shape == conv_w.shape
    assert t.equal(round_trip, conv_w), 'swap is its own inverse'

    # Load into a real nn.ConvTranspose2d to confirm the shape is accepted.
    ct = nn.ConvTranspose2d(in_channels=IC, out_channels=OC, kernel_size=(KH, KW), bias=False)
    assert ct.weight.shape == ct_w.shape, (
        f'nn.ConvTranspose2d expects {tuple(ct.weight.shape)}, our constructed weight is {tuple(ct_w.shape)}'
    )
    with t.no_grad():
        ct.weight.data.copy_(ct_w)
    assert t.equal(ct.weight, ct_w), 'copy_ should make the layer hold our exact weight'

    # Sanity — the constructed convT forward runs without shape errors.
    y = t.randn(1, IC, 4, 6, generator=rng)
    out = ct(y)
    # OH = (4 - 1) * 1 - 0 + (KH - 1) + 1 = 6;  OW = 6 + (KW-1) = 10
    assert out.shape == (1, OC, 6, 10), f'unexpected forward shape {tuple(out.shape)}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_conv_to_convT_weight(conv_weight: Tensor) -> Tensor:
    return conv_weight.transpose(0, 1).contiguous()
```

**Why `transpose(0, 1)` and nothing else.** The ONLY layout difference between Conv2d's `(OC, IC, KH, KW)` and ConvTranspose2d's `(IC, OC, KH, KW)` is the first two axes — the spatial axes are identical. So a single channel-axis transpose is the whole conversion.

**Why `.contiguous()`.** `.transpose` returns a non-contiguous view (same storage, swapped strides). Many downstream consumers — including `nn.Parameter` storage round-trips and CUDA kernels — expect contiguous tensors. Forcing contiguity here makes the constructed tensor a drop-in replacement.

**Why this is NOT enough to *compute* a transposed conv.** The axis-swap alone gives you the right LAYOUT, but a true ConvT-via-Conv2d equivalence ALSO needs a spatial flip on KH/KW and a `K-1` zero pad on the input (see the `convT-as-flipped-padded-conv` atom). ex2 here is only about the layout transformation; flipping is a separate skill.

**Difference from ex1.** ex1 *read* the two layouts — given modules, report their weight shapes and label axis 0. ex2 *produces* — given one layout, construct the other. Recognising the difference vs. effecting the difference are two different cognitive operations.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()